In [1]:
import os
import numpy as np
import nibabel as nib
from glob import glob

# Define paths
base_path = "/Users/songsifan/Downloads/Datasets/Med3d/Med3d_Others/AbdAutoPet/OWT-diffusion"
train_path = os.path.join(base_path, "Training/mask")
test_path = os.path.join(base_path, "Test/mask")

def calculate_organ_volumes(path):
    # Get all nii.gz files
    files = glob(os.path.join(path, "*.nii.gz"))
    
    # Initialize volume counters
    organ_voxels = {0: [], 1: [], 2: [], 3: [], 4: []}
    
    for file in files:
        # Load nii file
        img = nib.load(file)
        data = img.get_fdata()
        
        # Check if only contains allowed values
        unique_vals = np.unique(data)
        assert all(val in [0,1,2,3,4] for val in unique_vals), f"File {file} contains invalid values: {unique_vals}"
        
        # Get voxel dimensions
        voxel_size = np.prod(img.header.get_zooms())
        
        # Count volumes
        for organ_id in range(5):
            organ_volume = np.sum(data == organ_id) * voxel_size
            organ_voxels[organ_id].append(organ_volume)
    
    # Calculate means
    organ_means = {k: np.mean(v) if len(v)>0 else 0 for k,v in organ_voxels.items()}
    return organ_means

print("Processing Training data...")
train_volumes = calculate_organ_volumes(train_path)
print("\nTraining set average organ volumes (mm³):")
for organ_id, volume in train_volumes.items():
    print(f"Organ {organ_id}: {volume:.2f}")

print("\nProcessing Test data...")
test_volumes = calculate_organ_volumes(test_path)
print("\nTest set average organ volumes (mm³):")
for organ_id, volume in test_volumes.items():
    print(f"Organ {organ_id}: {volume:.2f}")


Processing Training data...

Training set average organ volumes (mm³):
Organ 0: 49070362.96
Organ 1: 1657221.81
Organ 2: 319009.71
Organ 3: 259484.45
Organ 4: 74145.08

Processing Test data...

Test set average organ volumes (mm³):
Organ 0: 49033966.04
Organ 1: 1657447.24
Organ 2: 332670.16
Organ 3: 280965.12
Organ 4: 75175.44


In [13]:
# Calculate token allocations using different methods
total_volume = sum(train_volumes.values())
total_tokens = 200

# Function to calculate and adjust token allocations
def calculate_tokens(volumes, method='normal'):
    tokens_per_organ = {}
    
    if method == 'normal':
        # Use direct volume ratios
        for organ_id, volume in volumes.items():
            ratio = volume / total_volume
            tokens = int(round(ratio * total_tokens))
            tokens_per_organ[organ_id] = tokens
            
    elif method == 'log':
        # Use log ratios
        log_volumes = {k: np.log10(v) for k,v in volumes.items()}
        total_log = sum(log_volumes.values())
        for organ_id, log_vol in log_volumes.items():
            ratio = log_vol / total_log
            tokens = int(round(ratio * total_tokens))
            tokens_per_organ[organ_id] = tokens
            
    elif method == 'sqrt':
        # Use square root ratios
        sqrt_volumes = {k: np.sqrt(v) for k,v in volumes.items()}
        total_sqrt = sum(sqrt_volumes.values())
        for organ_id, sqrt_vol in sqrt_volumes.items():
            ratio = sqrt_vol / total_sqrt
            tokens = int(round(ratio * total_tokens))
            tokens_per_organ[organ_id] = tokens
            
    elif method == 'cbrt':
        # Use cube root ratios
        cbrt_volumes = {k: np.cbrt(v) for k,v in volumes.items()}
        total_cbrt = sum(cbrt_volumes.values())
        for organ_id, cbrt_vol in cbrt_volumes.items():
            ratio = cbrt_vol / total_cbrt
            tokens = int(round(ratio * total_tokens))
            tokens_per_organ[organ_id] = tokens
            
    elif method == 'sqrt4':
        # Use 4th root ratios
        sqrt4_volumes = {k: v**(1/4) for k,v in volumes.items()}
        total_sqrt4 = sum(sqrt4_volumes.values())
        for organ_id, sqrt4_vol in sqrt4_volumes.items():
            ratio = sqrt4_vol / total_sqrt4
            tokens = int(round(ratio * total_tokens))
            tokens_per_organ[organ_id] = tokens
    
    # Adjust to ensure total is exactly total_tokens
    total_allocated = sum(tokens_per_organ.values())
    if total_allocated != total_tokens:
        tokens_per_organ[0] += (total_tokens - total_allocated)
        
    return tokens_per_organ

# Calculate using different methods
normal_tokens = calculate_tokens(train_volumes, 'normal')
log_tokens = calculate_tokens(train_volumes, 'log')
sqrt_tokens = calculate_tokens(train_volumes, 'sqrt')
cbrt_tokens = calculate_tokens(train_volumes, 'cbrt')
sqrt4_tokens = calculate_tokens(train_volumes, 'sqrt4')

print("\nToken allocation per organ (normal ratio):")
for organ_id, num_tokens in normal_tokens.items():
    print(f"Organ {organ_id}: {num_tokens} tokens ({(train_volumes[organ_id]/total_volume)*100:.1f}% of total volume)")

print("\nToken allocation per organ (log ratio):")
for organ_id, num_tokens in log_tokens.items():
    log_volumes = {k: np.log10(v) for k,v in train_volumes.items()}
    total_log = sum(log_volumes.values())
    print(f"Organ {organ_id}: {num_tokens} tokens ({(log_volumes[organ_id]/total_log)*100:.1f}% of log volume)")

# print("\nToken allocation per organ (square root ratio):")
# for organ_id, num_tokens in sqrt_tokens.items():
#     sqrt_volumes = {k: np.sqrt(v) for k,v in train_volumes.items()}
#     total_sqrt = sum(sqrt_volumes.values())
#     print(f"Organ {organ_id}: {num_tokens} tokens ({(sqrt_volumes[organ_id]/total_sqrt)*100:.1f}% of sqrt volume)")

print("\nToken allocation per organ (cube root ratio):")
for organ_id, num_tokens in cbrt_tokens.items():
    cbrt_volumes = {k: np.cbrt(v) for k,v in train_volumes.items()}
    total_cbrt = sum(cbrt_volumes.values())
    print(f"Organ {organ_id}: {num_tokens} tokens ({(cbrt_volumes[organ_id]/total_cbrt)*100:.1f}% of cbrt volume)")

print("\nToken allocation per organ (4th root ratio):")
for organ_id, num_tokens in sqrt4_tokens.items():
    sqrt4_volumes = {k: v**(1/4) for k,v in train_volumes.items()}
    total_sqrt4 = sum(sqrt4_volumes.values())
    print(f"Organ {organ_id}: {num_tokens} tokens ({(sqrt4_volumes[organ_id]/total_sqrt4)*100:.1f}% of 4th root volume)")



Token allocation per organ (normal ratio):
Organ 0: 192 tokens (95.5% of total volume)
Organ 1: 6 tokens (3.2% of total volume)
Organ 2: 1 tokens (0.6% of total volume)
Organ 3: 1 tokens (0.5% of total volume)
Organ 4: 0 tokens (0.1% of total volume)

Token allocation per organ (log ratio):
Organ 0: 52 tokens (25.9% of log volume)
Organ 1: 42 tokens (20.9% of log volume)
Organ 2: 37 tokens (18.5% of log volume)
Organ 3: 36 tokens (18.2% of log volume)
Organ 4: 33 tokens (16.4% of log volume)

Token allocation per organ (cube root ratio):
Organ 0: 111 tokens (55.6% of cbrt volume)
Organ 1: 36 tokens (18.0% of cbrt volume)
Organ 2: 21 tokens (10.4% of cbrt volume)
Organ 3: 19 tokens (9.7% of cbrt volume)
Organ 4: 13 tokens (6.4% of cbrt volume)

Token allocation per organ (4th root ratio):
Organ 0: 92 tokens (45.9% of 4th root volume)
Organ 1: 39 tokens (19.7% of 4th root volume)
Organ 2: 26 tokens (13.0% of 4th root volume)
Organ 3: 25 tokens (12.4% of 4th root volume)
Organ 4: 18 toke